# Data Cleaning Pipeline cic 2018 Dataset

Cleaning and balancing pipeline for 2018

**Steps:**
1. Convert large CSV to Parquet in chunks
2. Analyze label distribution
3. Balance dataset (downsample Benign + large attack classes)
4. Export balanced dataset

In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import dask.dataframe as dd
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, recall_score
from sklearn.utils.class_weight import compute_class_weight
import os, glob, gc, time, warnings, joblib
warnings.filterwarnings('ignore')


## Convert Large CSV to Parquet in Chunks
- Optimize memory usage (downcast data types)


In [ ]:
DATA_DIR   = "/home/huyho/earth_predict_env/dataset/Dataset_cic_2018"
OUTPUT_DIR = "/home/huyho/earth_predict_env/dataset/Dataset_cic_2018/parquet_data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

files = sorted(glob.glob(os.path.join(DATA_DIR, "**/*.csv"), recursive=True))

def reduce_memory(df):
    for col in df.columns:
        col_type = df[col].dtype
        if col_type != object:
            c_min, c_max = df[col].min(), df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
            else:
                df[col] = df[col].astype(np.float32)
    return df

def fix_dtypes(df):
    for col in df.columns:
        if col in ('Label', 'Timestamp'):
            continue
        if df[col].dtype == object:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    return df

CHUNK_SIZE = 500_000

for f in files:
    name = os.path.basename(f).replace('.csv', '.parquet')
    output_path = os.path.join(OUTPUT_DIR, name)
    if os.path.exists(output_path):
        continue
    processed_chunks = []
    for i, chunk in enumerate(pd.read_csv(f, low_memory=False, chunksize=CHUNK_SIZE), 1):
        chunk = fix_dtypes(chunk)
        chunk = reduce_memory(chunk)
        processed_chunks.append(chunk)
    df_tmp = pd.concat(processed_chunks, ignore_index=True)
    df_tmp = df_tmp[df_tmp['Label'] != 'Label']
    del processed_chunks; gc.collect()
    df_tmp = fix_dtypes(df_tmp)
    df_tmp.to_parquet(output_path, index=False)
    del df_tmp; gc.collect()


Found 11 CSV files


## Balance Dataset and Export Balanced File

**Balancing strategy:**
- Downsample **Benign** 
- Downsample large attack classes
- Keep small attack classes unchanged 


In [ ]:
OUTPUT_DIR = "/home/huyho/earth_predict_env/dataset/Dataset_cic_2018/parquet_data"
parquet_files = sorted(glob.glob(os.path.join(OUTPUT_DIR, "02-*.parquet")))
if not parquet_files:
    parquet_files = sorted([f for f in glob.glob(os.path.join(OUTPUT_DIR, "*.parquet"))
                            if "hdbscan_test_set" not in f])

df_dask = dd.read_parquet(parquet_files)

TARGET_BENIGN = 50_000
total_benign  = df_dask[df_dask['Label'] == 'Benign'].shape[0].compute()

frac = TARGET_BENIGN / total_benign
df_benign_sampled = df_dask[df_dask['Label'] == 'Benign'].sample(frac=frac, random_state=42).compute()
if len(df_benign_sampled) > TARGET_BENIGN:
    df_benign_sampled = df_benign_sampled.sample(n=TARGET_BENIGN, random_state=42)

df_attack = df_dask[df_dask['Label'] != 'Benign'].compute()

attacks_to_reduce = [
    'DDOS attack-HOIC', 'DDoS attacks-LOIC-HTTP', 'DDOS attack-LOIC-UDP',
    'DoS attacks-Hulk', 'DoS attacks-SlowHTTPTest', 'DoS attacks-GoldenEye',
    'DoS attacks-Slowloris', 'Bot', 'FTP-BruteForce', 'SSH-Bruteforce', 'Infilteration',
]
TARGET_ATTACK = 25_000

df_attacks_to_reduce = df_attack[df_attack['Label'].isin(attacks_to_reduce)]
df_other_attacks     = df_attack[~df_attack['Label'].isin(attacks_to_reduce)]

downsampled_attacks = []
for attack_label in attacks_to_reduce:
    df_at = df_attacks_to_reduce[df_attacks_to_reduce['Label'] == attack_label]
    original_count = len(df_at)
    if original_count > TARGET_ATTACK:
        df_at = df_at.sample(n=TARGET_ATTACK, random_state=42)
    downsampled_attacks.append(df_at)

df_balanced = pd.concat([df_benign_sampled] + downsampled_attacks + [df_other_attacks], ignore_index=True)

del df_dask, df_benign_sampled, df_attack, df_attacks_to_reduce, df_other_attacks, downsampled_attacks
gc.collect()
label_counts  = df_balanced['Label'].value_counts()
label_percent = (label_counts / label_counts.sum() * 100).sort_values(ascending=False)
label_counts  = label_counts[label_percent.index]

for lbl in label_percent.index:
    print(f"{lbl:<35} {label_counts[lbl]:>15,} {label_percent[lbl]:>10.2f}")


Benign                                       49,999      18.96
DDOS attack-HOIC                             25,000       9.48
DDoS attacks-LOIC-HTTP                       25,000       9.48
DoS attacks-Hulk                             25,000       9.48
DoS attacks-SlowHTTPTest                     25,000       9.48
DoS attacks-GoldenEye                        25,000       9.48
FTP-BruteForce                               25,000       9.48
SSH-Bruteforce                               25,000       9.48
Infilteration                                25,000       9.48
DoS attacks-Slowloris                        10,990       4.17
DDOS attack-LOIC-UDP                          1,730       0.66
Brute Force -Web                                611       0.23
Brute Force -XSS                                230       0.09
SQL Injection                                    87       0.03


In [12]:
BALANCED_OUTPUT_DIR = "/home/huyho/earth_predict_env/dataset/Dataset_cic_2018"
os.makedirs(BALANCED_OUTPUT_DIR, exist_ok=True)

parquet_output = os.path.join(BALANCED_OUTPUT_DIR, "cic2018_balanced_dataset.parquet")
df_balanced.to_parquet(parquet_output, index=False)
